In [ ]:
from diffusers import StableDiffusion3Pipeline
import torch
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

    # ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

In [ ]:
from pathlib import Path
import gc
import pandas as pd
from tqdm import tqdm
import shutil

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_tsr_samples_tester_comp")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_pt_samples_tester_comp")
PROMPTS_FILE = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/data_files/compbench_colors.csv")

REPLICA_EXCHANGE = True
LAM_VALUES = [1.15]
N_SAMPLES = 4

gc.collect()
torch.cuda.empty_cache()

Blip

In [ ]:
from transformers import BlipProcessor, BlipForImageTextRetrieval

processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").cuda()

def score_blip(image, prompt):
    inputs = processor(image, prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        score = model(**inputs).itm_score
    return torch.softmax(score, dim=1)[0][1].item()  # probability of match

Unidet

In [ ]:
def score_unidet(image, prompt, target_class, target_count):
    detections = unidet_model.detect(image)
    detected = [d for d in detections if d["class"] == target_class 
                and d["score"] > CONFIDENCE_THRESHOLD]
    # reward being close to target count
    return -abs(len(detected) - target_count) / target_count

In [ ]:
# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")


replica_exchanges = [True, False]

lam_dirs = {}
for re in replica_exchanges:
	base = PT_TSR_DIR if re else TSR_DIR
	lam_dirs[re] = {l: base / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
	for d in lam_dirs[re].values():
		d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	
	for replica_exchange in replica_exchanges:
	
		for n in range(N_SAMPLES):

			output_dir = lam_dirs[replica_exchange][tsr_lam]

			if (output_dir / f"{idx:05d}.png").exists():
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED + n)

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=TSR_SIGMA,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images[0]

			blip_score  = score_blip(image, prompt)
			unidet_score = score_unidet(image, prompt)
			combined = blip_score + unidet_score  # or weighted
			candidates.append((combined, image))
			
		# pick best
		best_score, best_image = max(candidates, key=lambda x: x[0])
		best_image.save(output_dir / f"{idx:05d}.png")